# 4. Mecanismo de similaridade entre projetos

**Requisito da vaga:** *Apoiar o desenvolvimento do mecanismo de similaridade entre projetos*.

Baseline sem LLM: distância **Euclidiana normalizada** sobre os vetores de *features de projeto* escalados (`StandardScaler`). A candidata mais próxima no espaço padronizado recebe score mais alto em `[0, 1]`.

Implementação real: `python/ml_service/features.py` + `similarity.py`. Metodologia em `docs/similarity.md`.

> **Pré-requisito:** banco local com dados (`make db && make migrate && make smoke`)
> e o ML service rodando (`make ml-run &`) — ou apenas `make demo`.
>
> Carregar o helper compartilhado (bootstrap de imports, leitores de dados,
> URLs dos serviços) da primeira célula. `notebooks/common.py`.


In [1]:
import sys
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores')
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/notebooks')
import common
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
from ml_service.features import fit_design_features, DESIGN_FEATURES
from ml_service.similarity import similarity_scores

## Features de projeto (ordem estável)

As mesmas colunas numéricas da base histórica (Requisito 1) e do ELT (Requisito 2).

In [2]:
design = common.load_design_base()
feature_cols = common.design_features()
display(pd.DataFrame({'feature': feature_cols}))

,feature
0,rated_power_mva
1,hv_voltage_kv
2,lv_voltage_kv
3,frequency_hz
4,phase_count
5,impedance_percent
6,commissioning_year
7,no_load_loss_kw
8,load_loss_kw
9,total_mass_t


## Escalamento

Antes da distância, cada dimensão é padronizada (média 0, desvio 1). Assim `rated_power_mva` no padrão MVA e `commissioning_year` competem em escala equivalente.

In [3]:
design_records = [r.to_dict() for _, r in design.iterrows()]
model = fit_design_features(design_records)
pv = model.transform(design_records[0])
pd.DataFrame({'feature': model.columns, 'scaled_component': np.round(pv, 3)})

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


,feature,scaled_component
0,rated_power_mva,-0.678
1,hv_voltage_kv,-0.415
2,lv_voltage_kv,-0.875
3,frequency_hz,0.378
4,phase_count,0.000
5,impedance_percent,-0.698
6,commissioning_year,-0.212
7,no_load_loss_kw,-0.599
8,load_loss_kw,-0.528
9,total_mass_t,-0.761


## Similaridade computada localmente (mesmo algoritmo do serviço)

Alvo: **TR-001**. Identificamos os candidatos mais próximos e os scores; o alvo é excluído dos candidatos (sem auto-match).

In [4]:
target_id = 'TR-001'
candidates = [r for r in design_records if r['transformer_id'] != target_id]
scores = similarity_scores(design_records[0], candidates, model)
pd.DataFrame(scores[:6], columns=['transformer_id', 'score'])

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/util

,transformer_id,score
0,TR-018,0.5967
1,TR-039,0.5797
2,TR-033,0.5425
3,TR-037,0.4940
4,TR-003,0.4708
5,TR-038,0.4512


## Comparando com o serviço ML

Confere se o resultado local bate com `POST /similar` (fora do processo Python — o MESMO código roda no serviço).

In [5]:
local = dict(similarity_scores(design_records[0], candidates, model)[:5])
remote = common.similar_for(target_id, top_k=5).set_index('transformer_id')['score'].to_dict()
pd.DataFrame({'local': local, 'servico_ml': remote}).round(4)

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/util

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/util

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/util

,local,servico_ml
TR-018,0.5967,0.5982
TR-039,0.5797,0.5823
TR-033,0.5425,0.5452
TR-037,0.4940,0.4963
TR-003,0.4708,0.4736


## Baseline vs. distância bruta

O score `1/(1+distância)` é monótono: quanto menor a distância, maior o score — legível e comparável entre alvos.

In [6]:
raw = [(rid, np.linalg.norm(model.transform(design_records[0]) - model.transform(r)))
       for rid, r in [(c['transformer_id'], c) for c in candidates[:5]]]
pd.DataFrame(raw, columns=['transformer_id', 'distancia_bruta']).assign(
    score=lambda d: (1 / (1 + d['distancia_bruta'])).round(4))

/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/.venv/lib/python3.11/site-packages/sklearn/util

,transformer_id,distancia_bruta,score
0,TR-002,3.623143,0.2163
1,TR-003,1.124146,0.4708
2,TR-004,1.687649,0.3721
3,TR-005,2.097215,0.3229
4,TR-006,4.899786,0.1695


## Conclusão

- Mecanismo baseline rápido, determinístico e sem LLM — adequado para embasar propostas e reuso de engenharia.
- Resultados do serviço Python e do cálculo local coincidem (mesmo código).
- Evolução possível: embeddings específicos de domínio ou similaridade multi-critério (documentado como próximo passo, fora de escopo).